# 02 — Kill-chain investigation by identity

When the **Cross-plan CWPP kill chain** Sentinel rule fires (or you have a
suspect identity from another source), this notebook reconstructs everything
the principal touched across CWPP plans, ordered by time, and projects it
onto MITRE ATT&CK.


In [ ]:
"""Shared setup — run me first.
Connects to Log Analytics via DefaultAzureCredential (uses az login, env vars, or managed identity).
"""
import os
import pandas as pd
from datetime import timedelta
from azure.identity import DefaultAzureCredential
from azure.monitor.query import LogsQueryClient, LogsQueryStatus

WORKSPACE_ID = os.environ.get("LAW_WORKSPACE_ID")  # GUID of the Log Analytics workspace
assert WORKSPACE_ID, "Set LAW_WORKSPACE_ID env var (workspace customerId GUID, not full ARM id)"

client = LogsQueryClient(DefaultAzureCredential())

def kql(query: str, hours: int = 24) -> pd.DataFrame:
    """Run a KQL query against the workspace and return the first table as a DataFrame."""
    resp = client.query_workspace(WORKSPACE_ID, query, timespan=timedelta(hours=hours))
    if resp.status == LogsQueryStatus.PARTIAL:
        print("WARNING: partial result -", resp.partial_error)
        tables = resp.partial_data
    elif resp.status == LogsQueryStatus.SUCCESS:
        tables = resp.tables
    else:
        raise RuntimeError(f"Query failed: {resp}")
    if not tables:
        return pd.DataFrame()
    t = tables[0]
    return pd.DataFrame(t.rows, columns=t.columns)

pd.set_option("display.max_colwidth", 120)
print("Workspace:", WORKSPACE_ID)


## Set the suspect identity

In [ ]:
SUSPECT = "attacker@example.com"   # UPN, SP appId, or object-id substring
HOURS   = 24


## All MDC alerts naming this identity

In [ ]:
df = kql(f"""
SecurityAlert
| where TimeGenerated > ago({HOURS}h)
| where ProviderName == "Azure Security Center"
| where Entities has "{SUSPECT}" or CompromisedEntity has "{SUSPECT}"
| project TimeGenerated, AlertSeverity, AlertName, CompromisedEntity,
          Plan=tostring(parse_json(ExtendedProperties).["Defender plan"]),
          Description, ExtendedLinks
| order by TimeGenerated asc
""", hours=HOURS)
df


## Azure Activity Log — what did they do?

In [ ]:
df = kql(f"""
AzureActivity
| where TimeGenerated > ago({HOURS}h)
| where Caller has "{SUSPECT}"
| where ActivityStatusValue in ("Success", "Succeeded", "Started")
| project TimeGenerated, OperationNameValue, ResourceProviderValue, _ResourceId, CallerIpAddress, Caller
| order by TimeGenerated asc
""", hours=HOURS)
df


## Sign-in correlation (if Entra ID logs are in this workspace)

In [ ]:
df = kql(f"""
SigninLogs
| where TimeGenerated > ago({HOURS}h)
| where UserPrincipalName has "{SUSPECT}" or AppId has "{SUSPECT}"
| project TimeGenerated, UserPrincipalName, AppDisplayName, IPAddress,
          Location=tostring(parse_json(LocationDetails).countryOrRegion),
          ResultType, RiskState, RiskLevelAggregated, ConditionalAccessStatus
| order by TimeGenerated asc
""", hours=HOURS)
df


## MITRE timeline plot

In [ ]:
import matplotlib.pyplot as plt
df = kql(f"""
SecurityAlert
| where TimeGenerated > ago({HOURS}h)
| where ProviderName == "Azure Security Center"
| where Entities has "{SUSPECT}" or CompromisedEntity has "{SUSPECT}"
| mv-expand Tactics=parse_json(Tactics)
| project TimeGenerated, Tactic=tostring(Tactics), AlertName
""", hours=HOURS)
if not df.empty:
    df = df.sort_values("TimeGenerated")
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(df["TimeGenerated"], df["Tactic"], c="crimson", s=80)
    for _, row in df.iterrows():
        ax.annotate(row["AlertName"][:30], (row["TimeGenerated"], row["Tactic"]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
    plt.title(f"MITRE timeline for {SUSPECT}")
    plt.tight_layout(); plt.show()
else:
    print("No alerts match the suspect.")
